# 🌸 Machine Learning con Julio: K-Vecinos Más Cercanos (KNN)

¡Bienvenido de vuelta, Julio! 👋 Ha pasado un buen rato desde la última clase, así que hoy **primero repasamos** y luego **terminamos el proyecto de las flores**.

**Cómo usar este cuaderno:**
- Ejecuta las celdas **en orden, de arriba hacia abajo** (`Shift + Enter`).
- Donde veas `___` (tres guiones bajos) te toca completar a ti. Si ejecutas una celda con `___` sin completar, saldrá un `NameError`: es normal, significa que falta tu parte.
- Si te atoras, revisa la guía `guia_knn.md` que está en esta misma carpeta.

**Plan de hoy:**
0. 🧠 Repaso: ¿qué era KNN?
1. 📦 Cargar las flores Iris
2. ✂️ Separar los datos (el examen oculto)
3. 🤖 Entrenar al modelo KNN (el chismoso)
4. 📝 Calificar el examen
5. 🔮 El juego de la flor misteriosa
6. 🚀 Bonus: experimentar con `k`

---
## 0. 🧠 Repaso: ¿qué era KNN?

KNN sirve para **clasificar un objeto nuevo según cómo están clasificados los objetos que tiene más cerca**.

> **Analogía:** llega un alumno nuevo a la escuela y no sabes si le gusta el fútbol o los videojuegos. Miras a los **3 compañeros que más se le parecen** (edad, gustos, con quién se sienta...). Si 2 de 3 juegan fútbol, apuestas a que a él también le gusta el fútbol. ⚽

Los pasos de KNN siempre son los mismos:
1. **Medir la distancia** entre el objeto nuevo y todos los que ya conocemos.
2. **Quedarse con los `k` más cercanos** (los "vecinos").
3. **Votar**: la etiqueta que más se repita entre los vecinos es la predicción.

Antes de usar la librería, vamos a hacerlo **a mano** con Python normal para recordar cómo funciona por dentro.

### Calentamiento 1: ¿qué tan lejos están dos puntos?

La distancia "en línea recta" entre dos puntos `(x1, y1)` y `(x2, y2)` es:

$$\text{distancia} = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

💡 Pista: en Python, `** 0.5` es lo mismo que sacar raíz cuadrada.

In [ ]:
def distancia(punto_a, punto_b):
    x1, y1 = punto_a
    x2, y2 = punto_b
    # TODO: escribe la fórmula de la distancia
    return ___

# Prueba: la distancia entre (0, 0) y (3, 4) debe ser 5.0
print(distancia((0, 0), (3, 4)))

### Calentamiento 2: KNN a mano con frutas 🍎🍋

Tenemos frutas descritas por **dos características**: peso (gramos) y acidez (del 1 al 10).
Encontramos una fruta misteriosa que pesa **150 g** y tiene acidez **3**. ¿Es manzana o limón?

In [ ]:
# Cada fruta es una tupla: (peso, acidez, etiqueta)
frutas_conocidas = [
    (170, 2, "manzana"),
    (160, 3, "manzana"),
    (180, 2, "manzana"),
    (90, 9, "limón"),
    (100, 8, "limón"),
    (110, 9, "limón"),
]

fruta_misteriosa = (150, 3)
k = 3

# Paso 1: medir la distancia a cada fruta conocida
distancias = []
for peso, acidez, etiqueta in frutas_conocidas:
    d = distancia(fruta_misteriosa, (peso, acidez))
    distancias.append((d, etiqueta))

# Paso 2: ordenar de la más cercana a la más lejana y quedarnos con las k primeras
distancias.sort()
vecinos = distancias[:___]
print("Vecinos más cercanos:", vecinos)

# Paso 3: votar (¡usamos un diccionario para contar, como en la Fase 2!)
votos = {}
for d, etiqueta in vecinos:
    votos[etiqueta] = votos.get(etiqueta, 0) + 1
print("Votos:", votos)

ganador = max(votos, key=votos.get)
print(f"🍽️ La fruta misteriosa probablemente es: {ganador}")

✅ **¡Eso es KNN completo!** Medir → ordenar → votar.

🤔 **Pregunta para pensar:** ¿por qué crees que se suele elegir un `k` **impar** (1, 3, 5...)? *(Pista: piensa qué pasa con 2 votos contra 2.)*

Ahora sí, vamos con datos de verdad y con la librería **Scikit-Learn**, que hace todo esto por nosotros (y mucho más rápido).

---
## 1. 📦 Cargar el set de datos clásico

Scikit-Learn ya viene con algunos datasets "de juguete" listos para usar. Vamos a importar el de las **flores Iris** y meterlo en un **DataFrame** de Pandas para verlo como una tabla.

Cada flor tiene 4 medidas (en centímetros): largo y ancho del **sépalo**, y largo y ancho del **pétalo**.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

# 1. Cargamos el dataset clásico de flores
iris = ___()

# 2. Lo convertimos a una tabla de Pandas
datos_flores = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# 3. Le agregamos la columna con la respuesta correcta (la especie de flor)
datos_flores['especie'] = iris.___

# ¡Miramos cómo se ve nuestra tabla! (solo las primeras 5 filas)
datos_flores.head()

📌 En la columna `especie` los números significan:

| Número | Especie |
|:-:|---|
| `0` | Setosa |
| `1` | Versicolor |
| `2` | Virginica |

La IA trabaja con números, no con palabras; por eso las especies vienen "codificadas".

Exploremos un poco más la tabla antes de entrenar nada:

In [ ]:
# ¿Cuántas filas (flores) y columnas tiene la tabla?
print("Tamaño de la tabla (filas, columnas):", datos_flores.___)

# ¿Cuántas flores hay de cada especie?
print(datos_flores['especie'].value_counts())

# Los nombres reales de las especies
print("Especies:", iris.target_names)

🤔 **Pregunta:** ¿hay la misma cantidad de flores de cada especie? ¿Por qué crees que eso es bueno para entrenar?

---
## 2. ✂️ Separar los datos (el examen oculto)

Igual que en clases anteriores: separamos el **80 %** de las flores para que la IA estudie sus medidas, y guardamos un **20 %** escondido para ponerle un examen.

- `X` (mayúscula) = las **características**: lo que la IA puede ver (las 4 medidas).
- `y` (minúscula) = la **etiqueta**: lo que queremos adivinar (la especie).

⚠️ Si la IA viera las respuestas del examen mientras estudia, sacaría 100 y no sabríamos si de verdad aprendió. ¡Por eso el examen va oculto!

In [ ]:
from sklearn.model_selection import train_test_split

# X son las características físicas: TODO menos la columna 'especie'
X = datos_flores.drop('especie', axis=1)

# y es lo que queremos adivinar (la especie)
y = datos_flores[___]

# Dividimos en entrenamiento (train) y prueba (test). ¿Qué número va para un 20 %?
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42)

print(f"🌸 Flores para estudiar: {len(X_train)}")
print(f"📝 Flores para el examen: {len(X_test)}")

---
## 3. 🤖 Entrenar al modelo KNN (el chismoso)

Creamos el modelo KNN. El parámetro `n_neighbors=3` significa que, cuando le demos una flor nueva, la IA buscará las **3 flores más parecidas** en su memoria, verá qué especie gana por mayoría de votos, y esa será su predicción.

Le decimos "el chismoso" porque **no aprende reglas**: solo se fija en lo que hacen sus vecinos. 😄 Por eso `fit` en KNN es casi instantáneo: lo único que hace es **memorizar** las flores de entrenamiento.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Creamos el modelo diciéndole que revise a los 3 vecinos más cercanos
modelo_vecinos = KNeighborsClassifier(n_neighbors=___)

# La IA memoriza la posición de las flores de ENTRENAMIENTO (¿cuáles dos variables van aquí?)
modelo_vecinos.fit(___, ___)

print("🤖 ¡Modelo KNN entrenado y listo para buscar vecinos!")

---
## 4. 📝 Evaluación y calificación del examen

Ahora sí: le damos al modelo las flores del examen **sin las respuestas** (`X_test`), y comparamos lo que adivinó con las respuestas reales (`y_test`).

In [ ]:
from sklearn.metrics import accuracy_score

# El modelo intenta adivinar las especies del examen
predicciones = modelo_vecinos.predict(___)

# Medimos su precisión: comparamos respuestas reales vs. predicciones
calificacion = accuracy_score(___, predicciones)
print(f"🎯 La precisión de KNN en el examen es de: {calificacion * 100:.2f}%")

🔎 Veamos el examen pregunta por pregunta, lado a lado:

In [ ]:
revision = pd.DataFrame({
    'respuesta_real': y_test.values,
    'prediccion_IA': predicciones,
})
revision['¿acertó?'] = revision['respuesta_real'] == revision['prediccion_IA']
revision.head(10)

🤔 **Pregunta:** si sacó 100 %, ¿significa que el modelo es perfecto para *cualquier* flor del mundo? ¿Por qué sí o por qué no?

---
## 5. 🔮 ¡El juego de la flor misteriosa!

Imagina que vas caminando por el bosque y descubres una flor Iris extraña. Vamos a **inventar sus medidas** y ver qué especie dice la IA que es.

📏 Orden de las medidas: `[largo sépalo, ancho sépalo, largo pétalo, ancho pétalo]`

💡 Metemos la flor en un `DataFrame` con **los mismos nombres de columna** que usamos para entrenar; así el modelo no se confunde (y no sale un aviso amarillo).

In [ ]:
# Inventa las medidas de tu flor (en cm). ¡Cámbialas y vuelve a ejecutar!
medidas = [5.1, 3.5, 1.4, 0.2]

flor_misteriosa = pd.DataFrame([medidas], columns=iris.feature_names)

# Le pedimos al modelo que busque a sus vecinos y vote
prediccion_numerica = modelo_vecinos.predict(___)[0]

# Traducimos el número al nombre real de la flor
nombre_flor = iris.target_names[prediccion_numerica]

print(f"🔍 Medidas de la flor: {medidas}")
print(f"🔮 Basado en sus vecinos más cercanos, la IA dice que es una: ¡Iris {nombre_flor.upper()}!")

### 🕵️ ¿Quiénes son los vecinos de tu flor?

El "chismoso" puede decirnos **a quién le preguntó**. `kneighbors` devuelve las distancias y las posiciones de los vecinos más cercanos dentro de `X_train`.

In [ ]:
distancias_vecinos, posiciones = modelo_vecinos.kneighbors(flor_misteriosa)

vecinos_flor = X_train.iloc[posiciones[0]].copy()
vecinos_flor['especie'] = iris.target_names[y_train.iloc[posiciones[0]]]
vecinos_flor['distancia'] = distancias_vecinos[0].round(2)
vecinos_flor

🎮 **Reto:** encuentra unas medidas para las que la IA diga **Versicolor** y otras para **Virginica**. (Pista: mira la tabla del paso 1 y fíjate sobre todo en los pétalos.)

In [ ]:
# Tu flor para Versicolor:
medidas_versicolor = [___, ___, ___, ___]

# Tu flor para Virginica:
medidas_virginica = [___, ___, ___, ___]

for medidas in [medidas_versicolor, medidas_virginica]:
    flor = pd.DataFrame([medidas], columns=iris.feature_names)
    especie = iris.target_names[modelo_vecinos.predict(flor)[0]]
    print(f"{medidas} -> Iris {especie}")

---
## 6. 🚀 Bonus: ¿cuántos vecinos conviene escuchar?

`k` es una **perilla** que nosotros elegimos (a esto se le llama *hiperparámetro*). Probemos varios valores y veamos cómo cambia la calificación.

In [ ]:
import matplotlib.pyplot as plt

valores_k = range(1, 26)
calificaciones = []

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_train, y_train)
    calificaciones.append(accuracy_score(y_test, modelo.predict(X_test)))

plt.plot(valores_k, calificaciones, marker='o')
plt.xlabel('k (número de vecinos)')
plt.ylabel('Precisión en el examen')
plt.title('¿Cuántos vecinos conviene escuchar?')
plt.grid(True)
plt.show()

🤔 **Preguntas para cerrar:**
1. ¿Qué pasaría con `k = 1`? ¿Y si `k` fuera igual al número total de flores de entrenamiento (120)?
2. Con Iris casi todos los `k` salen bien porque es un dataset fácil. ¿Se te ocurre un problema real donde KNN podría servir? (Ej.: recomendar canciones parecidas 🎵)
3. Explica con tus palabras, en una frase, cómo decide KNN.

✍️ Escribe tus respuestas aquí abajo (doble clic para editar):

- **1.**
- **2.**
- **3.**